# codex_ML_pipeline.ipynb

## ISY503 Assessment 2 — Broader ML Pipeline (TF2/Keras + scikit-learn)

**Purpose:** create a broader, submission-ready notebook that still follows the reflective, change-by-change style of `car_data_ML_pipeline.ipynb`, while explicitly covering the directions suggested by Dr Nandini:

- different regressors
- hidden-layer variations
- hyperparameter tuning
- optimizer comparisons
- batch-size comparisons
- training set sizes and proper data splits
- categorical encoding alternatives

**Main design choice in this refreshed notebook:** use **TF2/Keras + scikit-learn** as the implementation backbone so the broader experimentation is easier to organize, evaluate, and explain.

**Run policy:** execute the notebook **top-to-bottom, block-by-block**. Every later section depends on the cleaning, splitting, preprocessing, and helper functions defined earlier.


## Run Environment and Package Requirements

This notebook assumes a Python environment with these packages available:

- `numpy`
- `pandas`
- `matplotlib`
- `seaborn`
- `scikit-learn`
- `tensorflow` (TF2 / Keras)

If needed, install them before running:

```python
# Optional, run only if your notebook environment is missing packages.
# %pip install numpy pandas matplotlib seaborn scikit-learn tensorflow
```

**Engineering principle used throughout this notebook:** comments are intentionally explicit. The goal is not only to train models, but to make the reasoning behind each modelling choice easy to defend in the written manual.


In [ ]:
# [Change #1]: Imports, reproducibility, and notebook-wide display settings.
# This block centralizes everything needed for the experiments.

from pathlib import Path
import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder, StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

pd.options.display.max_columns = 100
pd.options.display.float_format = "{:.3f}".format
sns.set_theme(style="whitegrid", context="notebook")

print("TensorFlow version:", tf.__version__)
print("Random seed:", SEED)


In [ ]:
# [Change #2]: Load the local car dataset and define the schema explicitly.
# Use the local CSV first so the notebook is self-contained inside the Assessment 2 folder.

feature_names = [
    "symboling", "normalized-losses", "make", "fuel-type", "aspiration", "num-doors",
    "body-style", "drive-wheels", "engine-location", "wheel-base", "length", "width",
    "height", "weight", "engine-type", "num-cylinders", "engine-size", "fuel-system",
    "bore", "stroke", "compression-ratio", "horsepower", "peak-rpm", "city-mpg",
    "highway-mpg", "price"
]

DATA_PATH = Path("../data/car_data.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Expected local dataset at: {DATA_PATH.resolve()}")

raw_df = pd.read_csv(DATA_PATH, names=feature_names, header=None, encoding="latin-1")
raw_df = raw_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print("Loaded dataset shape:", raw_df.shape)
display(raw_df.head(3))


## Task 0 — Data Preparation and Evaluation Protocol

This notebook begins with the same core problem as the original notebook: the raw autos dataset includes `'?'` placeholders and mixed numeric / categorical types.

The refreshed difference is methodological:

- clean the dataset explicitly
- separate numeric and categorical features clearly
- define a **proper train / validation / test split**
- fit preprocessing steps using the **training set only**
- evaluate models using validation and test metrics, not only full-dataset replay

This makes the notebook broader and academically stronger than a pure single-dataset codelab run.


In [ ]:
# [Change #3]: Clean the raw dataset and define numeric / categorical feature groups.
# Key principle: do not train on missing labels. Feature missingness can be imputed later.

LABEL = "price"

numeric_feature_names = [
    "symboling", "normalized-losses", "wheel-base", "length", "width", "height", "weight",
    "engine-size", "horsepower", "peak-rpm", "city-mpg", "highway-mpg", "bore", "stroke",
    "compression-ratio"
]

categorical_feature_names = [
    c for c in feature_names if c not in numeric_feature_names + [LABEL]
]

df = raw_df.copy()

for col in numeric_feature_names + [LABEL]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Rows with missing target cannot be used for supervised regression.
df = df[df[LABEL].notna() & (df[LABEL] > 0)].copy()
df.reset_index(drop=True, inplace=True)

print("Cleaned dataset shape:", df.shape)
print("Numeric feature count:", len(numeric_feature_names))
print("Categorical feature count:", len(categorical_feature_names))
print("Remaining missing target values:", df[LABEL].isna().sum())
display(df[numeric_feature_names + [LABEL]].isna().sum().to_frame("missing_values"))


In [ ]:
# [Change #4]: Quick exploratory diagnostics.
# The goal is not full EDA; it is to confirm scale differences, target spread, and missingness patterns.

print("Target summary:")
display(df[LABEL].describe().to_frame().T)

print("Top missing-value columns before imputation:")
display(df.isna().sum().sort_values(ascending=False).head(10).to_frame("missing_values"))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df[LABEL], kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Price distribution")
axes[0].set_xlabel("price")

corr_cols = ["engine-size", "horsepower", "weight", "city-mpg", LABEL]
sns.heatmap(df[corr_cols].corr(numeric_only=True), annot=True, cmap="coolwarm", ax=axes[1])
axes[1].set_title("Quick numeric correlation check")
plt.tight_layout()
plt.show()


## Experiment Protocol

**Hard rule for this refreshed notebook:** model selection is based on validation performance, and final reporting uses the held-out test set.

- **Train split**: used to fit models and preprocessing statistics
- **Validation split**: used for architecture / hyperparameter comparisons
- **Test split**: used once final candidate models are selected

**Primary metrics:**

- `RMSE` — same units as the label (dollars), best for plain-English explanation
- `MAE` — robust and easy to interpret
- `R²` — useful supplementary fit measure

This is more defensible than evaluating on the full dataset used for training.


In [ ]:
# [Change #5]: Proper train / validation / test split.
# 60 / 20 / 20 is used here because it gives enough training data while preserving two honest evaluation sets.

train_df, temp_df = train_test_split(df, test_size=0.40, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED)

print(f"Train / Val / Test sizes: {len(train_df)} / {len(val_df)} / {len(test_df)}")
display(pd.DataFrame({
    "split": ["train", "val", "test"],
    "rows": [len(train_df), len(val_df), len(test_df)],
    "target_mean": [train_df[LABEL].mean(), val_df[LABEL].mean(), test_df[LABEL].mean()]
}))


In [ ]:
# [Change #6]: Build reusable preprocessing factories.
# These are fit on TRAIN only, then applied to validation and test.

def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def make_ordinal_encoder():
    return OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

def make_preprocessor(numeric_scaler="none", categorical_encoding="onehot"):
    if numeric_scaler == "standard":
        num_transformer = Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler())
        ])
    elif numeric_scaler == "minmax":
        num_transformer = Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", MinMaxScaler())
        ])
    else:
        num_transformer = Pipeline([
            ("imputer", SimpleImputer(strategy="mean"))
        ])

    if categorical_encoding == "onehot":
        cat_transformer = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", make_one_hot_encoder())
        ])
    elif categorical_encoding == "ordinal":
        cat_transformer = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", make_ordinal_encoder())
        ])
    else:
        raise ValueError("categorical_encoding must be 'onehot' or 'ordinal'")

    return ColumnTransformer([
        ("num", num_transformer, numeric_feature_names),
        ("cat", cat_transformer, categorical_feature_names)
    ])

def transform_split(preprocessor, train_part, val_part, test_part, feature_mode="all"):
    if feature_mode == "numeric":
        X_train_df = train_part[numeric_feature_names]
        X_val_df = val_part[numeric_feature_names]
        X_test_df = test_part[numeric_feature_names]
    elif feature_mode == "categorical":
        X_train_df = train_part[categorical_feature_names]
        X_val_df = val_part[categorical_feature_names]
        X_test_df = test_part[categorical_feature_names]
    else:
        X_train_df = train_part[numeric_feature_names + categorical_feature_names]
        X_val_df = val_part[numeric_feature_names + categorical_feature_names]
        X_test_df = test_part[numeric_feature_names + categorical_feature_names]

    X_train = preprocessor.fit_transform(X_train_df)
    X_val = preprocessor.transform(X_val_df)
    X_test = preprocessor.transform(X_test_df)

    y_train = train_part[LABEL].to_numpy(dtype="float32")
    y_val = val_part[LABEL].to_numpy(dtype="float32")
    y_test = test_part[LABEL].to_numpy(dtype="float32")

    return X_train.astype("float32"), X_val.astype("float32"), X_test.astype("float32"), y_train, y_val, y_test


In [ ]:
# [Change #7]: Shared model builders and evaluation helpers.
# This keeps the notebook broad without becoming repetitive and error-prone.

experiment_rows = []

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def add_result(task, family, model_name, feature_mode, numeric_scaler, categorical_encoding,
               optimizer_name, batch_size, hidden_units, notes,
               val_pred, test_pred, y_val, y_test):
    experiment_rows.append({
        "task": task,
        "family": family,
        "model": model_name,
        "feature_mode": feature_mode,
        "numeric_scaler": numeric_scaler,
        "categorical_encoding": categorical_encoding,
        "optimizer": optimizer_name,
        "batch_size": batch_size,
        "hidden_units": str(hidden_units),
        "val_rmse": rmse(y_val, val_pred),
        "test_rmse": rmse(y_test, test_pred),
        "val_mae": float(mean_absolute_error(y_val, val_pred)),
        "test_mae": float(mean_absolute_error(y_test, test_pred)),
        "val_r2": float(r2_score(y_val, val_pred)),
        "test_r2": float(r2_score(y_test, test_pred)),
        "pred_mean_test": float(np.mean(test_pred)),
        "label_mean_test": float(np.mean(y_test)),
        "notes": notes
    })

def make_optimizer(name, learning_rate):
    name = name.lower()
    if name == "adam":
        return keras.optimizers.Adam(learning_rate=learning_rate)
    if name == "rmsprop":
        return keras.optimizers.RMSprop(learning_rate=learning_rate)
    raise ValueError(f"Unsupported optimizer: {name}")

def build_linear_keras(input_dim, learning_rate=1e-2, optimizer_name="adam"):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(1, activation="linear")
    ])
    model.compile(
        optimizer=make_optimizer(optimizer_name, learning_rate),
        loss="mse",
        metrics=[keras.metrics.MeanAbsoluteError(name="mae")]
    )
    return model

def build_mlp(input_dim, hidden_units=(64,), learning_rate=1e-3, optimizer_name="adam", dropout=0.0):
    inputs = keras.Input(shape=(input_dim,))
    x = inputs
    for units in hidden_units:
        x = layers.Dense(units, activation="relu")(x)
        if dropout > 0:
            x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, activation="linear")(x)
    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=make_optimizer(optimizer_name, learning_rate),
        loss="mse",
        metrics=[keras.metrics.MeanAbsoluteError(name="mae")]
    )
    return model

def build_wide_deep(input_dim, deep_units=(64, 32), learning_rate=1e-3, optimizer_name="adam", dropout=0.0):
    # Keras equivalent of a wide-and-deep style regressor.
    # Wide branch: direct linear path from the encoded features.
    # Deep branch: nonlinear path to learn interactions.
    inputs = keras.Input(shape=(input_dim,))
    wide = layers.Dense(1, activation="linear", name="wide_linear")(inputs)
    x = inputs
    for units in deep_units:
        x = layers.Dense(units, activation="relu")(x)
        if dropout > 0:
            x = layers.Dropout(dropout)(x)
    deep = layers.Dense(1, activation="linear", name="deep_output")(x)
    outputs = layers.Add(name="wide_plus_deep")([wide, deep])
    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=make_optimizer(optimizer_name, learning_rate),
        loss="mse",
        metrics=[keras.metrics.MeanAbsoluteError(name="mae")]
    )
    return model

def fit_keras_experiment(task, model_name, builder, X_train, y_train, X_val, y_val, X_test, y_test,
                        family, feature_mode, numeric_scaler, categorical_encoding,
                        optimizer_name, batch_size, hidden_units, notes, epochs=200):
    model = builder(X_train.shape[1])
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_mae", patience=15, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=7, min_lr=1e-5)
    ]
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=0,
        callbacks=callbacks
    )
    val_pred = model.predict(X_val, verbose=0).reshape(-1)
    test_pred = model.predict(X_test, verbose=0).reshape(-1)
    add_result(task, family, model_name, feature_mode, numeric_scaler, categorical_encoding,
               optimizer_name, batch_size, hidden_units, notes,
               val_pred, test_pred, y_val, y_test)
    return model, history

def fit_sklearn_experiment(task, model_name, estimator, X_train, y_train, X_val, y_val, X_test, y_test,
                           family, feature_mode, numeric_scaler, categorical_encoding,
                           optimizer_name, batch_size, hidden_units, notes):
    estimator.fit(X_train, y_train)
    val_pred = estimator.predict(X_val)
    test_pred = estimator.predict(X_test)
    add_result(task, family, model_name, feature_mode, numeric_scaler, categorical_encoding,
               optimizer_name, batch_size, hidden_units, notes,
               val_pred, test_pred, y_val, y_test)
    return estimator

def results_df():
    if not experiment_rows:
        return pd.DataFrame()
    return pd.DataFrame(experiment_rows).sort_values(["task", "val_rmse", "test_rmse"]).reset_index(drop=True)


## Task 1 — Numeric Features, No Normalization

This first experimental block mirrors the original assessment constraint: use numeric features **without normalization**.

Broader take in this refreshed version:

- `LinearRegressor` baseline
- shallow DNN (`[64]`)
- deeper DNN (`[128, 64]`)
- `RandomForestRegressor` baseline for a non-neural comparison

This addresses model exploration more directly than a single DNN run.


In [ ]:
# [Change #8]: Numeric-only experiments without normalization.

numeric_raw_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="mean"))
])

X_train_num = numeric_raw_preprocessor.fit_transform(train_df[numeric_feature_names]).astype("float32")
X_val_num = numeric_raw_preprocessor.transform(val_df[numeric_feature_names]).astype("float32")
X_test_num = numeric_raw_preprocessor.transform(test_df[numeric_feature_names]).astype("float32")
y_train = train_df[LABEL].to_numpy(dtype="float32")
y_val = val_df[LABEL].to_numpy(dtype="float32")
y_test = test_df[LABEL].to_numpy(dtype="float32")

# 1. Linear baseline
fit_sklearn_experiment(
    task="Task 1",
    model_name="LinearRegression",
    estimator=LinearRegression(),
    X_train=X_train_num, y_train=y_train, X_val=X_val_num, y_val=y_val, X_test=X_test_num, y_test=y_test,
    family="linear",
    feature_mode="numeric",
    numeric_scaler="none",
    categorical_encoding="none",
    optimizer_name="not_applicable",
    batch_size="not_applicable",
    hidden_units="-",
    notes="Classic linear numeric baseline. Tests whether a simple additive relationship is sufficient."
)

# 2. Shallow numeric DNN
fit_keras_experiment(
    task="Task 1",
    model_name="Keras DNN [64]",
    builder=lambda input_dim: build_mlp(input_dim, hidden_units=(64,), learning_rate=1e-2, optimizer_name="adam", dropout=0.0),
    X_train=X_train_num, y_train=y_train, X_val=X_val_num, y_val=y_val, X_test=X_test_num, y_test=y_test,
    family="dnn",
    feature_mode="numeric",
    numeric_scaler="none",
    categorical_encoding="none",
    optimizer_name="adam",
    batch_size=16,
    hidden_units=(64,),
    notes="Shallow nonlinear numeric model. Main question: does one hidden layer beat the linear baseline?",
    epochs=200
)

# 3. Deeper numeric DNN
fit_keras_experiment(
    task="Task 1",
    model_name="Keras DNN [128, 64]",
    builder=lambda input_dim: build_mlp(input_dim, hidden_units=(128, 64), learning_rate=1e-3, optimizer_name="adam", dropout=0.10),
    X_train=X_train_num, y_train=y_train, X_val=X_val_num, y_val=y_val, X_test=X_test_num, y_test=y_test,
    family="dnn",
    feature_mode="numeric",
    numeric_scaler="none",
    categorical_encoding="none",
    optimizer_name="adam",
    batch_size=16,
    hidden_units=(128, 64),
    notes="Deeper numeric model. Tests whether extra depth improves fit or simply adds complexity on a small dataset.",
    epochs=200
)

# 4. Random forest baseline
fit_sklearn_experiment(
    task="Task 1",
    model_name="RandomForestRegressor",
    estimator=RandomForestRegressor(n_estimators=300, random_state=SEED, min_samples_split=2, max_features=0.75),
    X_train=X_train_num, y_train=y_train, X_val=X_val_num, y_val=y_val, X_test=X_test_num, y_test=y_test,
    family="tree_ensemble",
    feature_mode="numeric",
    numeric_scaler="none",
    categorical_encoding="none",
    optimizer_name="not_applicable",
    batch_size="not_applicable",
    hidden_units="-",
    notes="Non-neural comparator. Useful because tree ensembles can model nonlinear numeric relationships without scaling."
)

display(results_df().query("task == 'Task 1'")[[
    "model", "val_rmse", "test_rmse", "val_mae", "test_mae", "pred_mean_test", "label_mean_test", "notes"
]])


## Task 2 — Numeric Features with Normalization, Optimizers, and Batch Sizes

The original notebook focused strongly on normalization behaviour. This refreshed block keeps that focus, but makes the experimentation more structured:

- compare **StandardScaler** vs **MinMaxScaler**
- compare **Adam** vs **RMSProp**
- compare **batch size 16 vs 32**

This satisfies Dr Nandini's suggested take more directly than a single normalized rerun.


In [ ]:
# [Change #9]: Structured medium-depth hyperparameter exploration on normalized numeric features.

for scaler_name in ["standard", "minmax"]:
    numeric_scaled_preprocessor = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler() if scaler_name == "standard" else MinMaxScaler())
    ])

    X_train_scaled = numeric_scaled_preprocessor.fit_transform(train_df[numeric_feature_names]).astype("float32")
    X_val_scaled = numeric_scaled_preprocessor.transform(val_df[numeric_feature_names]).astype("float32")
    X_test_scaled = numeric_scaled_preprocessor.transform(test_df[numeric_feature_names]).astype("float32")

    for optimizer_name, learning_rate in [("adam", 1e-3), ("rmsprop", 1e-3)]:
        for batch_size in [16, 32]:
            fit_keras_experiment(
                task="Task 2",
                model_name=f"Numeric MLP [64] | {scaler_name} | {optimizer_name} | bs={batch_size}",
                builder=lambda input_dim, opt=optimizer_name, lr=learning_rate: build_mlp(
                    input_dim,
                    hidden_units=(64,),
                    learning_rate=lr,
                    optimizer_name=opt,
                    dropout=0.0
                ),
                X_train=X_train_scaled, y_train=y_train, X_val=X_val_scaled, y_val=y_val, X_test=X_test_scaled, y_test=y_test,
                family="dnn",
                feature_mode="numeric",
                numeric_scaler=scaler_name,
                categorical_encoding="none",
                optimizer_name=optimizer_name,
                batch_size=batch_size,
                hidden_units=(64,),
                notes="Normalized numeric MLP. Medium-depth search over scaler, optimizer, and batch size.",
                epochs=200
            )

display(results_df().query("task == 'Task 2'")[[
    "model", "numeric_scaler", "optimizer", "batch_size", "val_rmse", "test_rmse", "val_mae", "test_mae"
]])


## Task 3 — Categorical Features Only: One-Hot vs Ordinal

This block directly addresses a broader categorical exploration.

Encodings covered:

- **one-hot encoding**
- **ordinal encoding**

Models covered:

- linear baseline
- DNN (`[64]`)

Interpretive question:

> Are the categorical variables informative on their own, and does the choice of encoding materially affect the result?


In [ ]:
# [Change #10]: Categorical-only modelling with one-hot vs ordinal encoding.

for encoding_name in ["onehot", "ordinal"]:
    if encoding_name == "onehot":
        cat_transformer = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", make_one_hot_encoder())
        ])
    else:
        cat_transformer = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", make_ordinal_encoder())
        ])

    X_train_cat = cat_transformer.fit_transform(train_df[categorical_feature_names]).astype("float32")
    X_val_cat = cat_transformer.transform(val_df[categorical_feature_names]).astype("float32")
    X_test_cat = cat_transformer.transform(test_df[categorical_feature_names]).astype("float32")

    fit_sklearn_experiment(
        task="Task 3",
        model_name=f"Categorical Linear | {encoding_name}",
        estimator=Ridge(alpha=1.0),
        X_train=X_train_cat, y_train=y_train, X_val=X_val_cat, y_val=y_val, X_test=X_test_cat, y_test=y_test,
        family="linear",
        feature_mode="categorical",
        numeric_scaler="not_applicable",
        categorical_encoding=encoding_name,
        optimizer_name="not_applicable",
        batch_size="not_applicable",
        hidden_units="-",
        notes="Categorical-only linear baseline. Ridge is used here for a stable encoded linear comparator."
    )

    fit_keras_experiment(
        task="Task 3",
        model_name=f"Categorical DNN [64] | {encoding_name}",
        builder=lambda input_dim: build_mlp(input_dim, hidden_units=(64,), learning_rate=1e-3, optimizer_name="adam", dropout=0.10),
        X_train=X_train_cat, y_train=y_train, X_val=X_val_cat, y_val=y_val, X_test=X_test_cat, y_test=y_test,
        family="dnn",
        feature_mode="categorical",
        numeric_scaler="not_applicable",
        categorical_encoding=encoding_name,
        optimizer_name="adam",
        batch_size=16,
        hidden_units=(64,),
        notes="Categorical-only nonlinear model. Tests whether encoding choice changes how useful the nominal variables are.",
        epochs=200
    )

display(results_df().query("task == 'Task 3'")[[
    "model", "categorical_encoding", "val_rmse", "test_rmse", "val_mae", "test_mae"
]])


## Task 4 — All Features Combined

This is the broadest combined-feature section.

Models covered here:

- DNN regressor on fully encoded combined features
- wide-and-deep style combined regressor (Keras equivalent of `DNNLinearCombinedRegressor`)
- `RandomForestRegressor` on the same combined feature space

Encodings covered:

- one-hot + standardized numeric features
- ordinal + standardized numeric features

This block is where the final recommended model should emerge.


In [ ]:
# [Change #11]: Combined-feature experiments, including a Keras wide-and-deep equivalent.

for encoding_name in ["onehot", "ordinal"]:
    combined_preprocessor = make_preprocessor(numeric_scaler="standard", categorical_encoding=encoding_name)
    X_train_all, X_val_all, X_test_all, y_train_all, y_val_all, y_test_all = transform_split(
        combined_preprocessor,
        train_df,
        val_df,
        test_df,
        feature_mode="all"
    )

    fit_keras_experiment(
        task="Task 4",
        model_name=f"Combined DNN [64, 32] | {encoding_name}",
        builder=lambda input_dim: build_mlp(input_dim, hidden_units=(64, 32), learning_rate=1e-3, optimizer_name="adam", dropout=0.10),
        X_train=X_train_all, y_train=y_train_all, X_val=X_val_all, y_val=y_val_all, X_test=X_test_all, y_test=y_test_all,
        family="dnn",
        feature_mode="all",
        numeric_scaler="standard",
        categorical_encoding=encoding_name,
        optimizer_name="adam",
        batch_size=16,
        hidden_units=(64, 32),
        notes="Combined-feature deep model. Strong baseline for the final all-feature comparison.",
        epochs=200
    )

    fit_keras_experiment(
        task="Task 4",
        model_name=f"Wide + Deep Combined | {encoding_name}",
        builder=lambda input_dim: build_wide_deep(input_dim, deep_units=(64, 32), learning_rate=1e-3, optimizer_name="adam", dropout=0.05),
        X_train=X_train_all, y_train=y_train_all, X_val=X_val_all, y_val=y_val_all, X_test=X_test_all, y_test=y_test_all,
        family="wide_and_deep",
        feature_mode="all",
        numeric_scaler="standard",
        categorical_encoding=encoding_name,
        optimizer_name="adam",
        batch_size=16,
        hidden_units=(64, 32),
        notes="Keras equivalent of a DNNLinearCombined-style model: linear wide branch + nonlinear deep branch.",
        epochs=200
    )

    fit_sklearn_experiment(
        task="Task 4",
        model_name=f"Combined RandomForest | {encoding_name}",
        estimator=RandomForestRegressor(n_estimators=400, random_state=SEED, min_samples_split=2, max_features=0.75),
        X_train=X_train_all, y_train=y_train_all, X_val=X_val_all, y_val=y_val_all, X_test=X_test_all, y_test=y_test_all,
        family="tree_ensemble",
        feature_mode="all",
        numeric_scaler="standard",
        categorical_encoding=encoding_name,
        optimizer_name="not_applicable",
        batch_size="not_applicable",
        hidden_units="-",
        notes="Combined-feature tree ensemble baseline for the final ranking."
    )

display(results_df().query("task == 'Task 4'")[[
    "model", "categorical_encoding", "val_rmse", "test_rmse", "val_mae", "test_mae", "notes"
]])


## Task 5 — Training Set Size Sensitivity

Dr Nandini's suggested direction explicitly mentions **training set sizes and data splits**.

This block isolates that idea directly.

Question:

> If the final candidate model is trained on smaller fractions of the available training set, how much does validation / test performance degrade?

This helps you discuss whether the model is data-hungry and whether the small dataset size is likely limiting performance.


In [ ]:
# [Change #12]: Training-size sensitivity using the strongest combined setup as a template.

size_preprocessor = make_preprocessor(numeric_scaler="standard", categorical_encoding="onehot")
X_train_full, X_val_full, X_test_full, y_train_full, y_val_full, y_test_full = transform_split(
    size_preprocessor,
    train_df,
    val_df,
    test_df,
    feature_mode="all"
)

fractions = [0.50, 0.70, 1.00]
for frac in fractions:
    subset_size = max(16, int(len(X_train_full) * frac))
    subset_idx = np.random.RandomState(SEED).choice(len(X_train_full), size=subset_size, replace=False)
    X_train_sub = X_train_full[subset_idx]
    y_train_sub = y_train_full[subset_idx]

    fit_keras_experiment(
        task="Task 5",
        model_name=f"Wide + Deep Combined | onehot | train_frac={frac:.2f}",
        builder=lambda input_dim: build_wide_deep(input_dim, deep_units=(64, 32), learning_rate=1e-3, optimizer_name="adam", dropout=0.05),
        X_train=X_train_sub, y_train=y_train_sub, X_val=X_val_full, y_val=y_val_full, X_test=X_test_full, y_test=y_test_full,
        family="wide_and_deep",
        feature_mode="all",
        numeric_scaler="standard",
        categorical_encoding="onehot",
        optimizer_name="adam",
        batch_size=16,
        hidden_units=(64, 32),
        notes=f"Training-size sensitivity run using {subset_size} training rows out of {len(X_train_full)}.",
        epochs=200
    )

display(results_df().query("task == 'Task 5'")[[
    "model", "val_rmse", "test_rmse", "val_mae", "test_mae", "notes"
]])


## Final Ranking and Submission-Oriented Summary

This closing section turns the notebook from an exploration log into a submission-ready experiment record.

What this final block should help answer:

1. Which model performed best on the validation set?
2. Did that result hold on the test set?
3. Did nonlinear models clearly beat linear ones?
4. Did one-hot or ordinal encoding work better?
5. Did feature representation matter more than simply increasing model depth?
6. Did more training data help?


In [ ]:
# [Change #13]: Aggregate the full experiment table and surface the final candidates.

all_results = results_df()
display(all_results)

print("Top 10 experiments by validation RMSE:")
display(all_results.nsmallest(10, "val_rmse")[[
    "task", "model", "feature_mode", "numeric_scaler", "categorical_encoding",
    "optimizer", "batch_size", "hidden_units", "val_rmse", "test_rmse", "test_mae"
]])

best_per_task = all_results.loc[all_results.groupby("task")["val_rmse"].idxmin()].sort_values("task")
print("Best experiment per task:")
display(best_per_task[[
    "task", "model", "val_rmse", "test_rmse", "val_mae", "test_mae", "notes"
]])

output_path = Path("./codex_experiment_summary.csv")
all_results.to_csv(output_path, index=False)
print(f"Saved experiment summary to: {output_path.resolve()}")


## Reflection Prompts for the Manual

Use the outputs above to answer the manual in a concise, evidence-based way.

Suggested reflection structure:

- **Task 1:** explain why the linear baseline matters and whether a shallow DNN justified itself
- **Task 2:** explain whether normalization improved convergence or final error, and whether optimizer / batch size changed the result materially
- **Task 3:** explain whether categorical features alone were informative, and whether one-hot or ordinal encoding worked better
- **Task 4:** identify the strongest overall combined model and justify it
- **Task 5:** discuss whether the small dataset size appears to be a limiting factor

**One safe high-level conclusion if the results behave as expected:**

> A simple linear regressor is a useful baseline, but the car-price problem is not purely linear. Wider model exploration showed that nonlinear models usually perform better, but the biggest improvements come from stronger feature representation, proper preprocessing, and disciplined evaluation rather than simply making the network deeper.
